<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 13.4: Human-in-the-Loop Review-Answers
## Adding Analyst Oversight to Your Agent

**Estimated Time**: 40-50 minutes

### Learning Objectives
- Use `interrupt()` to pause graph execution and wait for human input
- Resume paused graphs with `Command(resume=...)` to inject analyst decisions
- Configure `MemorySaver` checkpointer to persist graph state across pauses
- Use `thread_id` to manage independent investigation threads
- Implement escalation workflows for ambiguous email classifications

### What You'll Build
An extension of Lab 3's tiered analysis that pauses for human review when the agent isn't confident:

```
... → compile_report → needs_review? →(yes)→ human_review (INTERRUPT) → apply_decision → END
                                      →(no)→ auto_finalize → END
```

### Why Human-in-the-Loop?
No AI system is perfect. In security operations, false positives block legitimate emails and false negatives let threats through. HITL ensures that ambiguous cases get expert human judgment while routine cases flow automatically.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# Verify API key is set
assert os.getenv("ANTHROPIC_API_KEY"), "Please set ANTHROPIC_API_KEY in your .env file"
print("Environment loaded successfully!")

In [ ]:
import json
from typing import Optional

from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

import sys
sys.path.insert(0, "..")
from utils.email_loader import load_emails, format_email_for_analysis
from utils.mock_tools import check_url_reputation, check_sender_reputation, check_email_authentication
from utils.display_helpers import display_verdict, display_investigation_report, display_graph_mermaid

In [ ]:
emails = load_emails(small=False)
print(f"Loaded {len(emails)} emails")

---
## Step 1: Define the Graph State

We extend the Lab 3 `TriageState` with one new field: `analyst_decision`. This field will hold
the human analyst's decision when the graph pauses for review.

| Field | Type | Purpose |
|-------|------|---------|
| `email` | `dict` | Raw email data |
| `email_text` | `str` | Formatted email string for the LLM |
| `risk_level` | `str \| None` | Triage risk: high, medium, low |
| `classification` | `dict \| None` | Structured classification from the LLM |
| `investigation_notes` | `list[str]` | Evidence gathered during investigation |
| `final_report` | `dict \| None` | Compiled investigation report |
| `analyst_decision` | `dict \| None` | **NEW**: Human analyst's review decision |

In [ ]:
class TriageState(TypedDict):
    email: dict
    email_text: str
    risk_level: Optional[str]
    classification: Optional[dict]
    investigation_notes: list[str]
    final_report: Optional[dict]
    analyst_decision: Optional[dict]   # NEW: human analyst's decision

In [ ]:
class RiskAssessment(BaseModel):
    """Initial risk assessment for triage routing."""
    risk_level: str = Field(
        description="Risk level: 'high', 'medium', or 'low'"
    )
    reasoning: str = Field(
        description="Brief explanation of why this risk level was assigned"
    )
    key_concerns: list[str] = Field(
        description="Top concerns that influenced the risk rating",
        default_factory=list,
    )


class EmailClassification(BaseModel):
    """Structured output for email classification."""
    verdict: str = Field(
        description="Classification verdict: 'phishing', 'suspicious', or 'legitimate'"
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0",
        ge=0.0,
        le=1.0,
    )
    reasoning: str = Field(
        description="Brief explanation of why this classification was chosen"
    )
    indicators: list[str] = Field(
        description="List of specific phishing indicators found (or why it appears legitimate)",
        default_factory=list,
    )

In [ ]:
llm = ChatAnthropic(model="claude-opus-4-8", max_tokens=4096)
risk_llm = llm.with_structured_output(RiskAssessment)
classify_llm = llm.with_structured_output(EmailClassification)

---
## Starter Code from Lab 3: Tiered Analysis Nodes

The following cells contain the complete, working nodes from Lab 3. These form the base pipeline
that Lab 4 extends with human-in-the-loop review:

- `initial_triage` -- assigns a risk level (high / medium / low)
- `route_by_risk` -- routes to the appropriate investigation depth
- `deep_investigation` -- full analysis with tool calls for high-risk emails
- `light_review` -- moderate analysis for medium-risk emails
- `fast_pass` -- minimal analysis for low-risk emails
- `compile_report` -- produces the final structured report

In [ ]:
# ============================================================
# Starter Code from Lab 3 -- all nodes provided in full
# ============================================================

def initial_triage(state: TriageState) -> dict:
    """Perform initial risk assessment to determine investigation depth."""
    email_text = format_email_for_analysis(state["email"])

    prompt = f"""You are a SOC Level 1 analyst performing initial email triage.

Assess the risk level of this email:
- HIGH: Multiple strong phishing indicators (spoofed domain, urgency, suspicious links, failed auth)
- MEDIUM: Some concerning elements but not clearly malicious (unknown sender, unusual request)
- LOW: Appears routine/legitimate (known internal sender, normal business content, passing auth)

Email:
{email_text}
"""

    result = risk_llm.invoke(prompt)
    return {
        "email_text": email_text,
        "risk_level": result.risk_level.lower(),
        "investigation_notes": [f"Initial triage: {result.risk_level.upper()} risk - {result.reasoning}"],
    }


def route_by_risk(state: TriageState) -> str:
    """Route to different investigation depths based on risk level."""
    risk = state["risk_level"]
    if risk == "high":
        return "deep_investigation"
    elif risk == "medium":
        return "light_review"
    else:
        return "fast_pass"


def deep_investigation(state: TriageState) -> dict:
    """Full investigation for high-risk emails -- uses security tools."""
    email = state["email"]
    notes = list(state["investigation_notes"])
    notes.append("--- Deep Investigation ---")

    # Check sender reputation
    sender_result = check_sender_reputation.invoke({"email_address": email["from_address"]})
    notes.append(f"Sender check: {sender_result}")

    # Check URLs
    for url in email.get("urls", []):
        url_result = check_url_reputation.invoke({"url": url})
        notes.append(f"URL check: {url_result}")

    # Check email authentication
    headers = email.get("headers", {})
    auth_result = check_email_authentication.invoke({
        "from_address": email["from_address"],
        "spf": headers.get("spf", "none"),
        "dkim": headers.get("dkim", "none"),
        "dmarc": headers.get("dmarc", "none"),
    })
    notes.append(f"Auth check: {auth_result}")

    # LLM classification with all evidence
    evidence_summary = "\n".join(notes)
    prompt = f"""You are a senior SOC analyst. Based on the email AND the investigation evidence below,
classify this email.

Email:
{state['email_text']}

Investigation Evidence:
{evidence_summary}

Provide your final classification."""

    result = classify_llm.invoke(prompt)
    return {
        "classification": result.model_dump(),
        "investigation_notes": notes,
    }


def light_review(state: TriageState) -> dict:
    """Moderate investigation for medium-risk emails."""
    email = state["email"]
    notes = list(state["investigation_notes"])
    notes.append("--- Light Review ---")

    # Check sender reputation only
    sender_result = check_sender_reputation.invoke({"email_address": email["from_address"]})
    notes.append(f"Sender check: {sender_result}")

    # Check email authentication
    headers = email.get("headers", {})
    auth_result = check_email_authentication.invoke({
        "from_address": email["from_address"],
        "spf": headers.get("spf", "none"),
        "dkim": headers.get("dkim", "none"),
        "dmarc": headers.get("dmarc", "none"),
    })
    notes.append(f"Auth check: {auth_result}")

    # LLM classification
    evidence_summary = "\n".join(notes)
    prompt = f"""You are a SOC analyst. Based on the email and limited investigation evidence,
classify this email.

Email:
{state['email_text']}

Investigation Evidence:
{evidence_summary}

Provide your classification."""

    result = classify_llm.invoke(prompt)
    return {
        "classification": result.model_dump(),
        "investigation_notes": notes,
    }


def fast_pass(state: TriageState) -> dict:
    """Quick classification for low-risk emails."""
    notes = list(state["investigation_notes"])
    notes.append("--- Fast Pass (low risk) ---")

    prompt = f"""You are a SOC analyst. This email was triaged as low risk.
Perform a quick classification.

Email:
{state['email_text']}

Provide your classification."""

    result = classify_llm.invoke(prompt)
    return {
        "classification": result.model_dump(),
        "investigation_notes": notes,
    }


def compile_report(state: TriageState) -> dict:
    """Compile the final investigation report."""
    email = state["email"]
    classification = state["classification"]

    verdict = classification["verdict"]
    if verdict == "phishing":
        recommendation = "BLOCK and alert security team"
    elif verdict == "suspicious":
        recommendation = "QUARANTINE for manual review"
    else:
        recommendation = "ALLOW delivery"

    report = {
        "email_id": email["id"],
        "subject": email["subject"],
        "from": f"{email['from_display_name']} <{email['from_address']}>",
        "verdict": classification["verdict"],
        "confidence": classification["confidence"],
        "risk_level": state["risk_level"],
        "reasoning": classification["reasoning"],
        "evidence": state["investigation_notes"],
        "recommendation": recommendation,
    }

    return {"final_report": report}

print("Lab 3 starter code loaded: initial_triage, route_by_risk, deep_investigation, light_review, fast_pass, compile_report")

---
## Concept: Checkpointing with MemorySaver

For a graph to pause and resume, it needs to **save its state** somewhere. LangGraph uses a
**checkpointer** for this.

`MemorySaver` is the simplest checkpointer -- it stores graph state in Python memory (a dictionary).
This is perfect for development and testing. In production, you would use a persistent backend
like `SqliteSaver` or `PostgresSaver`.

Key ideas:
- Each graph execution is identified by a **`thread_id`** (a string you choose)
- The checkpointer saves the state at each step, so the graph can resume exactly where it left off
- Different `thread_id` values are completely independent -- like separate browser tabs

```python
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

# Each invocation needs a thread_id
config = {"configurable": {"thread_id": "investigation-001"}}
result = graph.invoke(initial_state, config=config)
```

---
## Concept: `interrupt()` and `Command(resume=)`

These are the two halves of LangGraph's human-in-the-loop mechanism:

### `interrupt(value)`
When a node calls `interrupt(value)`, the graph **pauses execution** and returns control to the
caller. The `value` is sent back so the caller (your notebook, API server, etc.) can display it
to the human analyst. The graph's state is saved by the checkpointer.

```python
def human_review(state):
    review_info = {"verdict": state["verdict"], "confidence": state["confidence"]}
    decision = interrupt(review_info)  # <-- PAUSES HERE
    return {"analyst_decision": decision}
```

### `Command(resume=value)`
To resume a paused graph, you call `graph.invoke(Command(resume=value), config=config)` with
the same `thread_id`. The `value` becomes the return value of the `interrupt()` call inside
the paused node, and execution continues from where it stopped.

```python
# Analyst makes a decision
result = graph.invoke(
    Command(resume={"action": "approve"}),
    config={"configurable": {"thread_id": "investigation-001"}},
)
```

---
## Step 2: Lab 4 New Nodes

Now we build the human-in-the-loop components:

1. **`needs_review`** -- routing function that decides whether a verdict needs human review
2. **`human_review`** -- node that pauses via `interrupt()` and waits for the analyst
3. **`apply_decision`** -- node that applies the analyst's decision to the report
4. **`auto_finalize`** -- node for high-confidence verdicts that skip human review

In [ ]:
def needs_review(state: TriageState) -> str:
    """Determine if a human analyst needs to review this verdict."""
    report = state["final_report"]

    # Cases that need human review:
    # 1. Suspicious verdicts (ambiguous)
    # 2. Low confidence (< 0.7)
    # 3. High-risk emails classified as legitimate (potential false negative)

    verdict = report["verdict"].lower()
    confidence = report["confidence"]
    risk = report["risk_level"].lower()

    if verdict == "suspicious":
        return "human_review"
    if confidence < 0.7:
        return "human_review"
    if risk == "high" and verdict == "legitimate":
        return "human_review"

    return "auto_finalize"

print("needs_review routing function defined")

In [ ]:
def human_review(state: TriageState) -> dict:
    """Pause execution and wait for human analyst decision."""
    report = state["final_report"]

    # Present the case to the analyst
    review_prompt = {
        "email_id": report["email_id"],
        "subject": report["subject"],
        "from": report["from"],
        "ai_verdict": report["verdict"],
        "ai_confidence": f"{report['confidence']:.0%}",
        "risk_level": report["risk_level"],
        "reasoning": report["reasoning"],
        "instructions": "Please review and choose: 'approve', 'override', or 'escalate'",
        "override_options": ["phishing", "suspicious", "legitimate"],
    }

    # This pauses the graph!
    decision = interrupt(review_prompt)

    return {"analyst_decision": decision}

print("human_review node defined (uses interrupt)")

In [ ]:
def apply_decision(state: TriageState) -> dict:
    """Apply the analyst's decision to the final report."""
    decision = state["analyst_decision"]
    report = dict(state["final_report"])

    action = decision.get("action", "approve")

    if action == "approve":
        report["analyst_review"] = "Approved by analyst"
    elif action == "override":
        new_verdict = decision.get("new_verdict", report["verdict"])
        report["original_verdict"] = report["verdict"]
        report["verdict"] = new_verdict
        report["analyst_review"] = f"Overridden to '{new_verdict}' by analyst"
        # Update recommendation based on new verdict
        report["recommendation"] = (
            "BLOCK and alert security team" if new_verdict == "phishing"
            else "QUARANTINE for manual review" if new_verdict == "suspicious"
            else "ALLOW delivery"
        )
    elif action == "escalate":
        report["analyst_review"] = "Escalated to Tier 2 / Incident Response"
        report["recommendation"] = "ESCALATED - Awaiting Tier 2 analysis"

    return {"final_report": report}

print("apply_decision node defined")

In [ ]:
def auto_finalize(state: TriageState) -> dict:
    """Auto-finalize verdicts that don't need review."""
    report = dict(state["final_report"])
    report["analyst_review"] = "Auto-finalized (high confidence)"
    return {"final_report": report}

print("auto_finalize node defined")

---
## Step 3: Build the Graph with Checkpointer

Now we assemble everything into a single graph. The key differences from Lab 3:

1. After `compile_report`, a **conditional edge** routes to either `human_review` or `auto_finalize`
2. `human_review` flows into `apply_decision` (after the analyst resumes)
3. We compile with a **`MemorySaver` checkpointer** so the graph can pause and resume

In [ ]:
workflow = StateGraph(TriageState)

# Lab 3 nodes
workflow.add_node("initial_triage", initial_triage)
workflow.add_node("deep_investigation", deep_investigation)
workflow.add_node("light_review", light_review)
workflow.add_node("fast_pass", fast_pass)
workflow.add_node("compile_report", compile_report)

# Lab 4 nodes (NEW)
workflow.add_node("human_review", human_review)
workflow.add_node("apply_decision", apply_decision)
workflow.add_node("auto_finalize", auto_finalize)

# Lab 3 edges
workflow.add_edge(START, "initial_triage")
workflow.add_conditional_edges(
    "initial_triage",
    route_by_risk,
    {"deep_investigation": "deep_investigation", "light_review": "light_review", "fast_pass": "fast_pass"},
)
workflow.add_edge("deep_investigation", "compile_report")
workflow.add_edge("light_review", "compile_report")
workflow.add_edge("fast_pass", "compile_report")

# Lab 4 edges (NEW) - after compile_report, check if review needed
workflow.add_conditional_edges(
    "compile_report",
    needs_review,
    {"human_review": "human_review", "auto_finalize": "auto_finalize"},
)
workflow.add_edge("human_review", "apply_decision")
workflow.add_edge("apply_decision", END)
workflow.add_edge("auto_finalize", END)

# Compile WITH checkpointer
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)
print("HITL graph compiled!")

In [ ]:
display_graph_mermaid(graph, "Lab 4: Human-in-the-Loop Review")

---
## Test 1: Auto-Finalized Case (High Confidence)

Let's start with an email that the agent should be highly confident about. EMAIL-001 is a
classic credential-harvesting phishing email with a spoofed Microsoft domain, urgency tactics,
and failing authentication headers. The agent should classify it as phishing with high
confidence, and it should flow straight through to `auto_finalize` without pausing.

In [ ]:
# High-confidence phishing email - should auto-finalize
email = [e for e in emails if e["id"] == "EMAIL-001"][0]
print(f"Testing: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print()

config = {"configurable": {"thread_id": "thread-001"}}

result = graph.invoke(
    {
        "email": email,
        "email_text": "",
        "risk_level": None,
        "classification": None,
        "investigation_notes": [],
        "final_report": None,
        "analyst_decision": None,
    },
    config=config,
)

print(f"Verdict: {result['final_report']['verdict']}")
print(f"Confidence: {result['final_report']['confidence']:.0%}")
print(f"Review: {result['final_report'].get('analyst_review', 'N/A')}")
print()
display_investigation_report(result["final_report"])

---
## Test 2: Human Review Case (Ambiguous Email)

Now let's test with EMAIL-049, a borderline email. This is an unsolicited "free trial" offer
from `cloudflare-partners.com` -- not obviously malicious, but not clearly legitimate either.
The ground truth is `suspicious`.

The agent should flag this for human review because:
- The verdict may be "suspicious" (ambiguous)
- Confidence may be low (borderline signals)

When the graph pauses at `interrupt()`, we'll see the review prompt and then resume with a decision.

In [ ]:
# Borderline email - should trigger human review
email = [e for e in emails if e["id"] == "EMAIL-049"][0]
print(f"Testing: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print()

config = {"configurable": {"thread_id": "thread-002"}}

result = graph.invoke(
    {
        "email": email,
        "email_text": "",
        "risk_level": None,
        "classification": None,
        "investigation_notes": [],
        "final_report": None,
        "analyst_decision": None,
    },
    config=config,
)

# The graph should have paused! Let's check
print("Graph paused for human review!")
print(f"Current state verdict: {result['final_report']['verdict']}")
print(f"Confidence: {result['final_report']['confidence']:.0%}")
print(f"Risk level: {result['final_report']['risk_level']}")

### Resume with Analyst Decision: APPROVE

The analyst reviews the case and decides to approve the AI's verdict. We resume the graph
using `Command(resume=...)` with the same `thread_id`.

In [ ]:
# Analyst approves the AI's verdict
result = graph.invoke(
    Command(resume={"action": "approve"}),
    config=config,
)

print(f"Final verdict: {result['final_report']['verdict']}")
print(f"Review: {result['final_report'].get('analyst_review')}")
print()
display_investigation_report(result["final_report"])

---
## Test 3: Override Scenario

Let's test the override flow with EMAIL-050, another borderline email (a recruiter outreach).
The ground truth is `suspicious`. This time, the analyst disagrees with the AI and overrides
the verdict to `phishing`.

Note that we use a **new `thread_id`** (`thread-003`) because this is a separate investigation.

In [ ]:
# Test with another borderline email - this time override
email = [e for e in emails if e["id"] == "EMAIL-050"][0]
print(f"Testing: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print()

config_override = {"configurable": {"thread_id": "thread-003"}}

result = graph.invoke(
    {
        "email": email,
        "email_text": "",
        "risk_level": None,
        "classification": None,
        "investigation_notes": [],
        "final_report": None,
        "analyst_decision": None,
    },
    config=config_override,
)

print(f"AI verdict: {result['final_report']['verdict']}")
print(f"AI confidence: {result['final_report']['confidence']:.0%}")
print("\nAnalyst overrides to 'phishing'...")

# Override
result = graph.invoke(
    Command(resume={"action": "override", "new_verdict": "phishing"}),
    config=config_override,
)

print(f"\nFinal verdict: {result['final_report']['verdict']}")
print(f"Original verdict: {result['final_report'].get('original_verdict', 'N/A')}")
print(f"Review: {result['final_report'].get('analyst_review')}")
print(f"Recommendation: {result['final_report'].get('recommendation')}")
print()
display_investigation_report(result["final_report"])

---
## Test 4: Escalation Scenario

Let's test the escalation flow. We'll use EMAIL-048, a security newsletter with a
`suspicious` ground truth. This time the analyst decides the case is too complex for
Tier 1 and escalates it to Tier 2 / Incident Response.

In [ ]:
# Test with another borderline email - escalate
email = [e for e in emails if e["id"] == "EMAIL-048"][0]
print(f"Testing: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print()

config_escalate = {"configurable": {"thread_id": "thread-004"}}

result = graph.invoke(
    {
        "email": email,
        "email_text": "",
        "risk_level": None,
        "classification": None,
        "investigation_notes": [],
        "final_report": None,
        "analyst_decision": None,
    },
    config=config_escalate,
)

print(f"AI verdict: {result['final_report']['verdict']}")
print(f"AI confidence: {result['final_report']['confidence']:.0%}")
print("\nAnalyst escalates to Tier 2...")

# Escalate
result = graph.invoke(
    Command(resume={"action": "escalate"}),
    config=config_escalate,
)

print(f"\nFinal verdict: {result['final_report']['verdict']}")
print(f"Review: {result['final_report'].get('analyst_review')}")
print(f"Recommendation: {result['final_report'].get('recommendation')}")
print()
display_investigation_report(result["final_report"])

---
## Test 5: Auto-Finalized Legitimate Email

Let's also verify that clearly legitimate emails auto-finalize. EMAIL-026 is a simple
internal meeting change notification -- the agent should pass it quickly with high confidence.

In [ ]:
# Clearly legitimate email - should auto-finalize
email = [e for e in emails if e["id"] == "EMAIL-026"][0]
print(f"Testing: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print()

config_legit = {"configurable": {"thread_id": "thread-005"}}

result = graph.invoke(
    {
        "email": email,
        "email_text": "",
        "risk_level": None,
        "classification": None,
        "investigation_notes": [],
        "final_report": None,
        "analyst_decision": None,
    },
    config=config_legit,
)

print(f"Verdict: {result['final_report']['verdict']}")
print(f"Confidence: {result['final_report']['confidence']:.0%}")
print(f"Review: {result['final_report'].get('analyst_review', 'N/A')}")
print()
display_investigation_report(result["final_report"])

---
## Checkpoint Checklist

- [ ] Graph compiles with MemorySaver checkpointer
- [ ] Mermaid diagram shows the review branch after compile_report
- [ ] High-confidence verdicts auto-finalize without pausing
- [ ] Ambiguous/low-confidence verdicts pause at the interrupt
- [ ] `Command(resume={"action": "approve"})` successfully resumes and approves
- [ ] `Command(resume={"action": "override", "new_verdict": "..."})` changes the verdict
- [ ] `Command(resume={"action": "escalate"})` escalates to Tier 2
- [ ] Different thread_ids maintain independent state

## What's Next?

In **Lab 5**, we'll build a full multi-agent SOC team with specialized agents (Header Analyst, Content Analyst, URL/Attachment Analyst) coordinated by a supervisor agent. This mirrors how real SOC teams divide labor.